# 在 Colab 运行本项目
本单元将：
- 挂载 Google Drive
- 指定项目目录并切换工作路径
- 安装除 PyTorch 之外的依赖（避免与 Colab 自带 PyTorch 冲突）
- 检查 CUDA/GPU 与 PyTorch
- 尝试运行 main.py --help

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [2]:
import os, sys
IN_COLAB = False
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print('IN_COLAB:', IN_COLAB)
if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)

IN_COLAB: True


ValueError: mount failed

In [4]:
# 设置项目目录：优先使用 Drive:/MyDrive/TransferAttack，否则使用 /content/TransferAttack
import os
drive_path = '/content/drive/MyDrive/TransferAttack'
local_path = '/content/TransferAttack'
PROJECT_DIR = drive_path if os.path.exists(drive_path) else local_path
os.makedirs(PROJECT_DIR, exist_ok=True)
print('PROJECT_DIR =', PROJECT_DIR)

PROJECT_DIR = /content/TransferAttack


In [5]:
# 切换到项目目录并浏览文件
import os
os.chdir(PROJECT_DIR)
print('CWD:', os.getcwd())
print('Files:', sorted(os.listdir())[:50])

CWD: /content/TransferAttack
Files: []


In [6]:
# 检查 PyTorch/CUDA
try:
    import torch, torchvision
    print('torch:', torch.__version__, 'torchvision:', torchvision.__version__)
    print('CUDA available:', torch.cuda.is_available(), 'device_count:', torch.cuda.device_count())
except Exception as e:
    print('PyTorch not available:', e)

torch: 2.9.0+cu126 torchvision: 0.24.0+cu126
CUDA available: True device_count: 1


In [ ]:
# 生成 Colab 专用依赖清单：排除 torch/torchvision 与各类 --index-url
import os, re
req_in = os.path.join(os.getcwd(), 'requirements.txt')
req_out = os.path.join(os.getcwd(), 'colab_requirements.txt')
keep = []
if os.path.exists(req_in):
    with open(req_in, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith('#'):
                continue
            if s.startswith('--index-url') or s.startswith('--extra-index-url'):
                continue
            if re.search(r'^(torch|torchvision)(\b|==|>=|<=|\+)', s):
                continue
            keep.append(s)
    with open(req_out, 'w', encoding='utf-8') as f:
        f.write('
'.join(keep) + ('
' if keep else ''))
    print('Wrote', req_out, 'with', len(keep), 'lines')
else:
    print('requirements.txt not found at', req_in)

In [ ]:
# 安装非 PyTorch 依赖
import os, sys, subprocess
req_out = os.path.join(os.getcwd(), 'colab_requirements.txt')
if os.path.exists(req_out) and os.path.getsize(req_out) > 0:
    code = subprocess.call([sys.executable, '-m', 'pip', 'install', '-q', '-r', req_out])
    print('pip exit code:', code)
else:
    print('No extra dependencies to install; skipping.')

In [ ]:
# 可选：验证 GPU 信息（Colab 运行时需启用 GPU）
import os
os.system('nvidia-smi')

In [ ]:
# 尝试运行 main.py --help
import os, sys, subprocess
if os.path.exists('main.py'):
    ret = subprocess.call([sys.executable, 'main.py', '--help'])
    print('main.py --help exit code:', ret)
else:
    print('main.py not found in', os.getcwd())